Requirements
- PyTorch
- Kagglehub
- Pandas
- Datasets

In [1]:
from datasets import load_dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import re
import torch
import numpy as np

/home/ghawkes/Documents/Projects/Chatbot/.venv_old/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Get the first two statements in the form of a dictionary
def get_two_resp(convo):
        responses = re.split(" '[ \n]*' ", convo)

        if len(responses) >= 2:
            return responses[0], responses[1]
        else:
             return None, None
        
def clean_str(str):
     str = re.sub("[^ a-zA-Z]+", "", str)
     str = re.sub("[ ]+", " ", str)
     str = str.strip()
     str = str.lower()
     str = re.sub(" [ ]+", " ", str)
     return str
            

def clean(df):
    dialog = df["dialog"]
    
    q_a = np.empty((len(dialog), 2), dtype=np.dtypes.StringDType) # Numpy array where first col is question and second is the answer

    i = 0
    for convo in dialog:
        first_q, first_a = get_two_resp(convo)
        if first_q == None:
              continue

        clean_q = clean_str(first_q)
        clean_a = clean_str(first_a) 

        if len(clean_q) > 0 and len(clean_a) > 0:
            q_a[i][0] = clean_q
            q_a[i][1] = clean_a
            i = i + 1
        
    
    # Remove empty rows at the end
    q_a = q_a[0:i]

    return q_a



In [3]:
# Set the path to the file you'd like to load
file_path = "test.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "thedevastator/dailydialog-unlock-the-conversation-potential-in",
  file_path,
)




/tmp/ipykernel_747/1609868290.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [4]:
qa_dataset = clean(df)

Tokenize the data

In [5]:
def validate_text(text):
    # Check for capital letters
    if bool(re.search(r'[A-Z]', text)):
        raise ValueError("Text must be all lowercase")
    if bool(re.search(r'[^a-z ]', text)):
        raise ValueError("Text must be all lowercase letters and spaces")
    if bool(re.search(r'  ', text)):
        raise ValueError("Text must not include double spaces")
    
    if len(text) <= 1:
        print(text)
        print("about to err")
    if text[0] == " " or text[len(text) - 1] == " ":
        print(text)
        print("is text")
        raise ValueError("Text must be stripped")


def load_token_dataset(dataset):
    start_token = 0
    end_token = 1
    token_map = {"PAD":0, "START_TOKEN":1, "END_TOKEN":2} # Converts token to id quickly
    id_map = ["PAD", "START_TOKEN", "END_TOKEN"] # Index is the token id. Converts id to string token


    token_id = 3 # Counter for new ID
    max_tokens = -1 # Track the most number of tokens needed for question or answer in the database
    for row in dataset:
        # Get the text from the question and answer
        questions = row[0]
        answers = row[1]
        validate_text(questions)
        validate_text(answers)

        question_strs = questions.split(" ")
        answer_strs = answers.split(" ")

        if len(question_strs) > max_tokens: 
            max_tokens = len(question_strs)
        if len(answer_strs) > max_tokens: 
            max_tokens = len(answer_strs)

        # Add each new unique question to token_map
        for t in question_strs:
            if token_map.get(t) == None:
                token_map[t] = token_id
                id_map.append(t)
                token_id = token_id + 1
        
        # Add each new unique answer to token_map
        for t in answer_strs:
            if token_map.get(t) == None:
                token_map[t] = token_id
                id_map.append(t)
                token_id = token_id + 1
    
    return token_map, id_map, max_tokens


# Tokenizes a string and pads with zeros to the longest sentence length plus two (for start/end tokens)
def tokenize(text, token_map, max_tokens_len):
    token_strs = text.lower().split(" ")
    if len(token_strs) > max_tokens_len + 2:
        raise ValueError("Text may not have more tokens than the max_tokens_len + 2")
    tokens = np.zeros(max_tokens_len + 2, dtype=np.int64) # Include two extra tokens for start and end
    tokens[0] = token_map["START_TOKEN"]
    i = 1
    for t in token_strs:
        if token_map.get(t) != None:
            tokens[i] = token_map.get(t)
            i = i + 1
        else:
            pass # Skip unknown inputs
        
    
    tokens[i] = token_map["END_TOKEN"]
    
    return np.array(tokens)

def tokenize_dataset(q_a, token_map, longest_sentence):
    rows, cols = np.shape(q_a)
    tokenized_inputs = np.zeros((rows, cols, longest_sentence + 2), dtype=np.int64) # Third dimen is longest_sentence + 2 to include start/end tokens

    for i, row in enumerate(q_a):
        tokenized_inputs[i][0] = tokenize(row[0], token_map, longest_sentence) # Returns a numpy array
        tokenized_inputs[i][1] = tokenize(row[1], token_map, longest_sentence) # Returns a numpy array

    return tokenized_inputs


In [6]:
token_map, id_map, max_tokens = load_token_dataset(qa_dataset)

tokenized_dataset = tokenize_dataset(qa_dataset, token_map, max_tokens)

In [7]:
MAX_TOKENS = max_tokens

In [8]:
# Convert tokens to tensors
tokenized_dataset = torch.from_numpy(tokenized_dataset)

Positional encoding - gives the model the original order of inputs

The next block uses code from 

In [9]:
def get_angles(pos, i, d_model):
    angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
    return pos * angle_rates

def positional_encoding(position, d_model):
    angle_rads = get_angles(np.arange(position)[:, np.newaxis], np.arange(d_model)[np.newaxis, :], d_model)
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2]) #for even positions using sin()
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2]) #for odd positions using cos()
    pos_encoding = angle_rads[np.newaxis,:]
    return pos_encoding

In [13]:
# Mask tokens. Any tokens beyond "END_TOKEN" will be masked
def create_padding_mask(seq):
    seq = torch.eq(torch.tensor([1,2,3,0,0]), 0).to(torch.int32)
    return seq[:, torch.newaxis, torch.newaxis, :]

In [ ]:
def create_lookahead_mask(size):
    return torch.ones((size, size))